# 行业校准DCF估值 - 交互式探索

本Notebook用于探索行业基准参数、可视化调整系数分布、分析估值敏感性。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import tushare as ts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from industry_dcf.utils.rate_limiter import RateLimiter
from industry_dcf.utils.shenwan_lookup import find_l3_by_code, get_all_l3_codes
from industry_dcf.utils.industry_data_fetcher import IndustryDataFetcher
from industry_dcf.utils.industry_dcf_calculator import IndustryDCFCalculator

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 初始化Tushare
TUSHARE_TOKEN = os.environ.get('TUSHARE_TOKEN', '')
assert TUSHARE_TOKEN, '请设置 TUSHARE_TOKEN 环境变量'
ts.set_token(TUSHARE_TOKEN)
pro = ts.pro_api()

## 1. 选择目标股票和行业

In [ ]:
# 输入目标股票代码
TS_CODE = '002001.SZ'  # 修改为你要分析的股票

# 查找行业
industry_info = find_l3_by_code(TS_CODE, pro)
if industry_info:
    print(f"股票: {TS_CODE}")
    print(f"行业: {industry_info['l1_name']} > {industry_info['l2_name']} > {industry_info['l3_name']}")
    print(f"L3代码: {industry_info['l3_code']}")
else:
    print(f'未找到 {TS_CODE} 的行业分类')

## 2. 获取行业数据并计算基准参数

In [ ]:
rate_limiter = RateLimiter()
fetcher = IndustryDataFetcher(pro, rate_limiter=rate_limiter)
calculator = IndustryDCFCalculator()

# 获取行业数据
l3_code = industry_info['l3_code']
industry_financials = fetcher.get_industry_financials(l3_code)

# 计算行业基准
benchmark = calculator.calculate_industry_benchmark(industry_financials)

print(f"行业公司数: {benchmark['company_count']}")
print(f"有效公司数: {benchmark['valid_company_count']}")
print(f"FCFF/营收中位数: {benchmark['fcff_rev_ratio']['median']:.4f}")
print(f"FCFF/营收 P25-P75: {benchmark['fcff_rev_ratio']['p25']:.4f} - {benchmark['fcff_rev_ratio']['p75']:.4f}")
print(f"FCFF增长率中位数: {benchmark['fcff_growth_rate']['median']:.2%}")
print(f"营收增长率中位数: {benchmark['revenue_growth_rate']['median']:.2%}")
print(f"行业成熟度: {benchmark['industry_maturity']}")

## 3. 行业内FCFF/营收比分布可视化

In [ ]:
# 收集所有公司的FCFF/营收比
company_ratios = {}
for ts_code, data in industry_financials.get('companies', {}).items():
    cf_df = pd.DataFrame(data.get('cashflow') or [])
    inc_df = pd.DataFrame(data.get('income') or [])
    if cf_df.empty:
        continue
    fcff_list = calculator._calc_company_fcff(cf_df, inc_df)
    rev_map = calculator._extract_annual_revenue(inc_df)
    ratios = []
    for f in fcff_list[-3:]:
        rev = rev_map.get(f['year'])
        if rev and rev > 0:
            ratio = f['fcff'] / rev
            if -1 < ratio < 1:
                ratios.append(ratio)
    if ratios:
        company_ratios[ts_code] = np.mean(ratios)

# Plot distribution
if company_ratios:
    values = list(company_ratios.values())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(values, bins=20, edgecolor='black', alpha=0.7)
    axes[0].axvline(np.median(values), color='red', linestyle='--', label=f'中位数: {np.median(values):.4f}')
    axes[0].set_xlabel('FCFF/营收比')
    axes[0].set_ylabel('公司数')
    axes[0].set_title(f'行业内FCFF/营收比分布 ({industry_info["l3_name"]})')
    axes[0].legend()
    
    # Bar chart for each company
    sorted_items = sorted(company_ratios.items(), key=lambda x: x[1])
    codes = [k.split('.')[0] for k, v in sorted_items]
    vals = [v for k, v in sorted_items]
    colors = ['green' if v > 0 else 'red' for v in vals]
    axes[1].barh(range(len(codes)), vals, color=colors, alpha=0.7)
    axes[1].axvline(np.median(values), color='red', linestyle='--')
    axes[1].set_yticks(range(len(codes)))
    axes[1].set_yticklabels(codes, fontsize=7)
    axes[1].set_xlabel('FCFF/营收比')
    axes[1].set_title('各公司FCFF/营收比')
    
    plt.tight_layout()
    plt.show()

## 4. 执行个股估值

In [ ]:
# 获取目标公司数据
company_data = fetcher.get_company_financials(TS_CODE)

# 获取市场数据
from industry_dcf.main import _fetch_market_data
market_data = _fetch_market_data(TS_CODE, pro, rate_limiter)

print(f"当前股价: {market_data['current_price']}")
print(f"总股本(万股): {market_data['total_shares']:,.0f}")
print(f"总市值(元): {market_data['market_cap']:,.0f}")

# 执行估值
result = calculator.calculate_company_valuation(
    ts_code=TS_CODE,
    industry_benchmark=benchmark,
    company_financials=company_data,
    market_data=market_data,
)

if 'error' in result:
    print(f'估值失败: {result["error"]}')
else:
    print(f"\n每股价值: {result['per_share_value']:.2f} 元")
    print(f"安全边际: {result['safety_margin_pct']:+.1f}%")
    print(f"调整系数α: {result['alpha']:.4f}")
    print(f"混合增长率: {result['blended_growth_rate']:.2%}")

## 5. 敏感性分析热力图

In [ ]:
sens = result.get('sensitivity', {})
if sens:
    wacc_vals = sens['wacc_values']
    growth_vals = sens['growth_values']
    matrix = np.array([[cell['per_share'] for cell in row] for row in sens['matrix']])
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Heatmap
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(len(growth_vals)))
    ax.set_xticklabels([f'{g:.0%}' for g in growth_vals])
    ax.set_yticks(range(len(wacc_vals)))
    ax.set_yticklabels([f'{w:.0%}' for w in wacc_vals])
    ax.set_xlabel('增长率')
    ax.set_ylabel('WACC')
    ax.set_title(f'每股价值敏感性分析 - {TS_CODE}')
    
    # Annotate cells
    for i in range(len(wacc_vals)):
        for j in range(len(growth_vals)):
            val = matrix[i, j]
            color = 'white' if abs(val - matrix.mean()) > matrix.std() else 'black'
            ax.text(j, i, f'{val:.1f}', ha='center', va='center', color=color, fontsize=9)
    
    # Mark current price
    plt.colorbar(im, label='每股价值(元)')
    plt.tight_layout()
    plt.show()

## 6. 预测期FCFF走势

In [ ]:
if 'projected_fcff' in result:
    proj = result['projected_fcff']
    years = [p['year'] for p in proj]
    fcff_vals = [p['fcff'] for p in proj]
    rev_vals = [p['revenue'] for p in proj]
    
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    ax1.bar(years, fcff_vals, alpha=0.6, color='steelblue', label='预测FCFF(万元)')
    ax1.set_xlabel('预测年份')
    ax1.set_ylabel('FCFF (万元)', color='steelblue')
    
    ax2 = ax1.twinx()
    ax2.plot(years, rev_vals, 'ro-', label='预测营收(万元)')
    ax2.set_ylabel('营收 (万元)', color='red')
    
    ax1.set_title(f'{TS_CODE} 预测期FCFF与营收 (α={result["alpha"]:.2f}, g={result["blended_growth_rate"]:.2%})')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.show()